## 1. Setup


In [ ]:
!pip install -q rasterio scikit-learn

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import tensorflow as tf

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    balanced_accuracy_score,
)

drive.mount('/content/drive')

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path('/content/drive/MyDrive/Lucy_CNN/supervised_cnn')
PATCH_DIR = PROJECT_DIR / 'Sentinel1_grid_patches'
CNN_DIR = PROJECT_DIR / 'CNN_damage'
CNN_DIR.mkdir(parents=True, exist_ok=True)

LABELS_PATH = CNN_DIR / 'port_damage_labels_compl.csv'
MODEL_PATH = CNN_DIR / 'port_surface_damage_cnn.keras'
SCALING_PATH = CNN_DIR / 'cnn_scaling.json'
THRESHOLD_PATH = CNN_DIR / 'cnn_threshold.json'
PATCH_PREDICTIONS_PATH = CNN_DIR / 'cnn_patch_predictions.csv'
PORT_RESULTS_PATH = CNN_DIR / 'cnn_port_surface_results.csv'

BAND_NAMES = [
    'pre_VV',
    'pre_VH',
    'post_VV',
    'post_VH',
    'change_VV',
    'change_VH',
]

N_CHANNELS = len(BAND_NAMES)
BATCH_SIZE = 8
MAX_EPOCHS = 30
LEARNING_RATE = 0.001
MIN_VALIDATION_RECALL = 0.80

print('Patch directory:', PATCH_DIR)
print('Labels:', LABELS_PATH)
print('CNN output directory:', CNN_DIR)


## 2. Load and validate labels


In [ ]:
if not LABELS_PATH.exists():
    raise FileNotFoundError(f'Label CSV not found: {LABELS_PATH}')

if not PATCH_DIR.exists():
    raise FileNotFoundError(f'Patch directory not found: {PATCH_DIR}')

labels = pd.read_csv(LABELS_PATH)

required_columns = {
    'sample_id', 'source_port_id', 'port_name', 'country', 'disaster',
    'north_offset_m', 'east_offset_m', 'image_file',
    'include_port_surface', 'surface_damage_label'
}

missing_columns = required_columns - set(labels.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

labels['include_port_surface'] = pd.to_numeric(
    labels['include_port_surface'], errors='coerce'
).fillna(0).astype(int)

labels['surface_damage_label'] = pd.to_numeric(
    labels['surface_damage_label'], errors='coerce'
).fillna(-1).astype(int)

model_df = labels[
    (labels['include_port_surface'] == 1)
    & (labels['surface_damage_label'].isin([0, 1]))
].copy()

model_df['raster_path'] = model_df['image_file'].apply(
    lambda filename: str(PATCH_DIR / filename)
)

missing_rasters = [
    path for path in model_df['raster_path']
    if not Path(path).exists()
]

if missing_rasters:
    raise FileNotFoundError(
        f'{len(missing_rasters)} labelled rasters are missing. '
        f'First example: {missing_rasters[0]}'
    )

if model_df.empty:
    raise ValueError('No verified intact/damaged port-surface patches are available.')

print('Total CSV rows:', len(labels))
print('Usable CNN patches:', len(model_df))
print('\nPatch class counts:')
print(
    model_df['surface_damage_label']
    .value_counts().sort_index()
    .rename(index={0: 'Intact', 1: 'Damaged'})
)
print('\nUsable port-events:', model_df['source_port_id'].nunique())


In [ ]:
port_summary = (
    model_df
    .groupby(['source_port_id', 'port_name', 'country', 'disaster'], as_index=False)
    .agg(
        number_patches=('surface_damage_label', 'size'),
        damaged_patches=('surface_damage_label', 'sum'),
        mean_label=('surface_damage_label', 'mean'),
    )
)
port_summary['port_class'] = (port_summary['mean_label'] >= 0.5).astype(int)
display(port_summary)


## 3. Check GeoTIFF inputs


In [ ]:
reference_shape = None
input_checks = []

for raster_path in model_df['raster_path']:
    with rasterio.open(raster_path) as src:
        shape = (src.height, src.width)
        input_checks.append({
            'raster_path': raster_path,
            'height': src.height,
            'width': src.width,
            'bands': src.count,
            'nodata': src.nodata,
            'descriptions': list(src.descriptions),
        })

        if reference_shape is None:
            reference_shape = shape

        if shape != reference_shape:
            raise ValueError(
                f'Raster dimensions differ: {raster_path} has {shape}, '
                f'expected {reference_shape}.'
            )

        if src.count != N_CHANNELS:
            raise ValueError(
                f'{raster_path} has {src.count} bands; expected {N_CHANNELS}.'
            )

PATCH_HEIGHT, PATCH_WIDTH = reference_shape
print('Patch dimensions:', PATCH_HEIGHT, 'x', PATCH_WIDTH)
print('Channels:', N_CHANNELS)
display(pd.DataFrame(input_checks).head())


## 4. Split by port

Neighbouring grid patches overlap, so all patches from a port stay in the same data split. With enough labelled ports, the split is stratified at port level. If the current dataset is too small for that, the notebook uses a port-level fallback split and prints a warning. Once the expanded dataset is added, the same code automatically uses the stronger stratified split.


In [ ]:
def build_group_split(dataframe, seed=SEED):
    group_table = (
        dataframe.groupby('source_port_id', as_index=False)
        .agg(
            mean_label=('surface_damage_label', 'mean'),
            n_patches=('surface_damage_label', 'size'),
        )
    )
    group_table['group_class'] = (group_table['mean_label'] >= 0.5).astype(int)
    class_counts = group_table['group_class'].value_counts().to_dict()

    enough_for_stratified = (
        len(group_table) >= 6
        and class_counts.get(0, 0) >= 3
        and class_counts.get(1, 0) >= 3
    )

    if enough_for_stratified:
        train_val_groups, test_groups = train_test_split(
            group_table,
            test_size=0.20,
            random_state=seed,
            stratify=group_table['group_class'],
        )

        train_groups, val_groups = train_test_split(
            train_val_groups,
            test_size=0.25,
            random_state=seed,
            stratify=train_val_groups['group_class'],
        )
        split_mode = 'stratified port-level split'

    else:
        warnings.warn(
            'Not enough intact and damaged port-events for a fully stratified '
            'three-way split. A port-level fallback split will be used so the '
            'pipeline can be tested. Final performance should be reported only '
            'after the expanded dataset is added.'
        )

        group_ids = group_table['source_port_id'].tolist()
        candidates = []

        for val_group in group_ids:
            for test_group in group_ids:
                if val_group == test_group:
                    continue

                train_group_ids = [
                    group for group in group_ids
                    if group not in {val_group, test_group}
                ]

                train_candidate = dataframe[
                    dataframe['source_port_id'].isin(train_group_ids)
                ]

                if train_candidate['surface_damage_label'].nunique() < 2:
                    continue

                val_candidate = dataframe[
                    dataframe['source_port_id'] == val_group
                ]
                test_candidate = dataframe[
                    dataframe['source_port_id'] == test_group
                ]

                score = abs(
                    train_candidate['surface_damage_label'].mean()
                    - dataframe['surface_damage_label'].mean()
                )

                bonus = (
                    int(val_candidate['surface_damage_label'].nunique() == 2)
                    + int(test_candidate['surface_damage_label'].nunique() == 2)
                )

                candidates.append(
                    (-bonus, score, val_group, test_group, train_group_ids)
                )

        if not candidates:
            raise ValueError(
                'A valid port-level split could not be created. '
                'Training requires both intact and damaged examples.'
            )

        _, _, val_group, test_group, train_group_ids = sorted(candidates)[0]
        train_groups = group_table[group_table['source_port_id'].isin(train_group_ids)]
        val_groups = group_table[group_table['source_port_id'] == val_group]
        test_groups = group_table[group_table['source_port_id'] == test_group]
        split_mode = 'fallback port-level split'

    train_ids = set(train_groups['source_port_id'])
    val_ids = set(val_groups['source_port_id'])
    test_ids = set(test_groups['source_port_id'])

    train = dataframe[dataframe['source_port_id'].isin(train_ids)].reset_index(drop=True)
    val = dataframe[dataframe['source_port_id'].isin(val_ids)].reset_index(drop=True)
    test = dataframe[dataframe['source_port_id'].isin(test_ids)].reset_index(drop=True)

    return train, val, test, split_mode


train_df, val_df, test_df, split_mode = build_group_split(model_df)

print('Split mode:', split_mode)
for name, frame in [('Training', train_df), ('Validation', val_df), ('Test', test_df)]:
    print('\n' + name)
    print('Ports:', frame['source_port_id'].nunique())
    print('Patches:', len(frame))
    print(frame['surface_damage_label'].value_counts().sort_index())
    print('Port IDs:', sorted(frame['source_port_id'].unique()))


In [ ]:
train_ports = set(train_df['source_port_id'])
val_ports = set(val_df['source_port_id'])
test_ports = set(test_df['source_port_id'])

assert train_ports.isdisjoint(val_ports)
assert train_ports.isdisjoint(test_ports)
assert val_ports.isdisjoint(test_ports)

if train_df['surface_damage_label'].nunique() < 2:
    raise ValueError('Training split must contain both intact and damaged examples.')

if val_df['surface_damage_label'].nunique() < 2:
    warnings.warn(
        'Validation contains only one class. Threshold optimisation will be '
        'disabled until more labelled port-events are added.'
    )

if test_df['surface_damage_label'].nunique() < 2:
    warnings.warn(
        'Test contains only one class. Current test metrics are not a robust '
        'estimate of generalisation.'
    )


## 5. Calculate channel scaling from training data only


In [ ]:
def calculate_scaling(dataframe):
    values_by_band = [[] for _ in range(N_CHANNELS)]

    for raster_path in dataframe['raster_path']:
        with rasterio.open(raster_path) as src:
            raster = src.read().astype(np.float32)
            nodata = src.nodata

        for band_index in range(N_CHANNELS):
            values = raster[band_index]
            valid = np.isfinite(values)
            if nodata is not None:
                valid &= values != nodata
            if np.any(valid):
                values_by_band[band_index].append(values[valid])

    lower = np.zeros(N_CHANNELS, dtype=np.float32)
    upper = np.ones(N_CHANNELS, dtype=np.float32)

    for band_index in range(N_CHANNELS):
        if not values_by_band[band_index]:
            raise ValueError(f'No valid values for {BAND_NAMES[band_index]}.')

        band_values = np.concatenate(values_by_band[band_index])
        lower[band_index] = np.percentile(band_values, 2)
        upper[band_index] = np.percentile(band_values, 98)

        if upper[band_index] <= lower[band_index]:
            upper[band_index] = lower[band_index] + 1.0

    return lower, upper


band_low, band_high = calculate_scaling(train_df)

scaling_table = pd.DataFrame({
    'band': BAND_NAMES,
    'p02': band_low,
    'p98': band_high,
})
display(scaling_table)

with open(SCALING_PATH, 'w') as file:
    json.dump({
        'band_names': BAND_NAMES,
        'p02': band_low.tolist(),
        'p98': band_high.tolist(),
        'patch_height': PATCH_HEIGHT,
        'patch_width': PATCH_WIDTH,
    }, file, indent=2)

print('Saved scaling:', SCALING_PATH)


## 6. Build TensorFlow datasets


In [ ]:
def read_scaled_patch(raster_path):
    if hasattr(raster_path, 'numpy'):
        raster_path = raster_path.numpy().decode('utf-8')
    elif isinstance(raster_path, bytes):
        raster_path = raster_path.decode('utf-8')

    with rasterio.open(raster_path) as src:
        raster = src.read().astype(np.float32)
        nodata = src.nodata

    raster = np.transpose(raster, (1, 2, 0))
    valid = np.isfinite(raster)

    if nodata is not None:
        valid &= raster != nodata

    scaled = np.zeros_like(raster, dtype=np.float32)

    for band_index in range(N_CHANNELS):
        band = raster[:, :, band_index]
        band_scaled = (
            (band - band_low[band_index])
            / (band_high[band_index] - band_low[band_index])
        )
        band_scaled = np.clip(band_scaled, 0.0, 1.0)
        scaled[:, :, band_index] = np.where(
            valid[:, :, band_index], band_scaled, 0.0
        )

    return scaled.astype(np.float32)


def tf_read_patch(raster_path, label):
    image = tf.py_function(
        read_scaled_patch,
        [raster_path],
        Tout=tf.float32,
    )
    image.set_shape([PATCH_HEIGHT, PATCH_WIDTH, N_CHANNELS])
    return image, tf.cast(label, tf.float32)


def make_dataset(dataframe, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe['raster_path'].astype(str).values,
        dataframe['surface_damage_label'].astype(np.float32).values,
    ))

    if shuffle:
        dataset = dataset.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    return (
        dataset
        .map(tf_read_patch, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

print(train_ds.element_spec)


## 7. Inspect a scaled input


In [ ]:
sample_row = train_df.iloc[0]
sample_patch = read_scaled_patch(sample_row['raster_path'])

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for band_index, ax in enumerate(axes.ravel()):
    ax.imshow(sample_patch[:, :, band_index], vmin=0, vmax=1)
    ax.set_title(BAND_NAMES[band_index])
    ax.axis('off')

fig.suptitle(f"Scaled CNN input: {sample_row['sample_id']}")
fig.tight_layout()
plt.show()


## 8. Build the CNN

The model follows the structure used in the course labs: convolution with ReLU, max pooling, a second convolutional layer, feature reduction, a dense layer and a sigmoid output. Rotation and flip augmentation are applied during training because the damage class should not depend on the compass orientation of the port within the patch.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    tf.keras.layers.RandomRotation(0.25, fill_mode='reflect', seed=SEED),
], name='augmentation')

inputs = tf.keras.layers.Input(
    shape=(PATCH_HEIGHT, PATCH_WIDTH, N_CHANNELS),
    name='sentinel1_patch',
)

x = augmentation(inputs)
x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(2)(x)
x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(2)(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='damage_probability')(x)

damage_cnn = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name='port_surface_damage_cnn',
)

damage_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
    ],
)

damage_cnn.summary()


## 9. Train the model


In [ ]:
class_counts = train_df['surface_damage_label'].value_counts().to_dict()
number_training_samples = len(train_df)

class_weight = {
    class_label: number_training_samples / (2 * class_counts[class_label])
    for class_label in [0, 1]
}

print('Class weights:', class_weight)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor='val_loss',
        save_best_only=True,
    ),
]

history = damage_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)


## 10. Training curves


In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(CNN_DIR / 'cnn_training_history.csv', index=False)
epoch_numbers = np.arange(1, len(history_df) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(epoch_numbers, history_df['loss'], label='Training')
axes[0].plot(epoch_numbers, history_df['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary cross-entropy loss')
axes[0].set_title('(a) Loss')
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.2)

axes[1].plot(epoch_numbers, history_df['accuracy'], label='Training')
axes[1].plot(epoch_numbers, history_df['val_accuracy'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('(b) Accuracy')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.2)

fig.suptitle('Port surface damage CNN training')
fig.tight_layout()

training_figure_path = CNN_DIR / 'cnn_training_curves.png'
fig.savefig(training_figure_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', training_figure_path)


## 11. Select the damage-probability threshold

Thresholds from 0.30 to 0.90 are compared on validation data only. The selected threshold must meet the target validation recall where possible and then maximises F1 score. If the current validation split contains only one class, threshold optimisation is postponed and 0.50 is retained until the expanded dataset is added.


In [ ]:
y_val = val_df['surface_damage_label'].to_numpy(dtype=int)
val_probability = damage_cnn.predict(val_ds, verbose=0).ravel()

threshold_rows = []
for threshold_value in np.arange(0.30, 0.91, 0.05):
    val_prediction = (val_probability >= threshold_value).astype(int)

    row = {
        'threshold': float(threshold_value),
        'accuracy': accuracy_score(y_val, val_prediction),
        'precision': precision_score(y_val, val_prediction, zero_division=0),
        'recall': recall_score(y_val, val_prediction, zero_division=0),
        'f1': f1_score(y_val, val_prediction, zero_division=0),
        'balanced_accuracy': np.nan,
    }

    if len(np.unique(y_val)) == 2:
        row['balanced_accuracy'] = balanced_accuracy_score(y_val, val_prediction)

    threshold_rows.append(row)

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results)

if val_df['surface_damage_label'].nunique() == 2:
    recall_candidates = threshold_results[
        threshold_results['recall'] >= MIN_VALIDATION_RECALL
    ]

    if not recall_candidates.empty:
        selected_row = recall_candidates.sort_values(
            ['f1', 'precision', 'balanced_accuracy'],
            ascending=False,
        ).iloc[0]
    else:
        warnings.warn(
            'No tested threshold met the target validation recall. '
            'Using the threshold with the highest validation F1 score.'
        )
        selected_row = threshold_results.sort_values(
            ['f1', 'recall', 'precision'],
            ascending=False,
        ).iloc[0]

    DAMAGE_THRESHOLD = float(selected_row['threshold'])

else:
    DAMAGE_THRESHOLD = 0.50
    warnings.warn(
        'Validation contains one class only. Using threshold 0.50 until '
        'the expanded dataset supports proper threshold selection.'
    )

threshold_results.to_csv(
    CNN_DIR / 'validation_threshold_results.csv',
    index=False,
)

with open(THRESHOLD_PATH, 'w') as file:
    json.dump({
        'damage_threshold': DAMAGE_THRESHOLD,
        'minimum_validation_recall': MIN_VALIDATION_RECALL,
    }, file, indent=2)

print('Selected damage threshold:', DAMAGE_THRESHOLD)


## 12. Evaluate on held-out test ports


In [ ]:
y_test = test_df['surface_damage_label'].to_numpy(dtype=int)
test_probability = damage_cnn.predict(test_ds, verbose=0).ravel()
test_prediction = (test_probability >= DAMAGE_THRESHOLD).astype(int)

test_metrics = {
    'accuracy': accuracy_score(y_test, test_prediction),
    'precision': precision_score(y_test, test_prediction, zero_division=0),
    'recall': recall_score(y_test, test_prediction, zero_division=0),
    'f1': f1_score(y_test, test_prediction, zero_division=0),
    'balanced_accuracy': np.nan,
    'auc': np.nan,
}

if len(np.unique(y_test)) == 2:
    test_metrics['balanced_accuracy'] = balanced_accuracy_score(y_test, test_prediction)
    test_metrics['auc'] = roc_auc_score(y_test, test_probability)

test_metrics_df = pd.DataFrame([test_metrics])
display(test_metrics_df)

test_metrics_df.to_csv(
    CNN_DIR / 'held_out_test_metrics.csv',
    index=False,
)

test_cm = confusion_matrix(y_test, test_prediction, labels=[0, 1])
print('Confusion matrix [intact, damaged]:')
print(test_cm)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
image = ax.imshow(test_cm)

for row in range(test_cm.shape[0]):
    for column in range(test_cm.shape[1]):
        ax.text(
            column,
            row,
            str(test_cm[row, column]),
            ha='center',
            va='center',
        )

ax.set_xticks([0, 1])
ax.set_xticklabels(['Intact', 'Damaged'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['Intact', 'Damaged'])
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('Held-out port confusion matrix')
fig.tight_layout()

confusion_path = CNN_DIR / 'held_out_confusion_matrix.png'
fig.savefig(confusion_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', confusion_path)


## 13. Predict all verified surface patches


In [ ]:
all_model_df = model_df.reset_index(drop=True)
all_model_ds = make_dataset(all_model_df, shuffle=False)
all_probability = damage_cnn.predict(all_model_ds, verbose=0).ravel()

prediction_columns = [
    'sample_id', 'source_port_id', 'port_name', 'country', 'disaster',
    'north_offset_m', 'east_offset_m', 'image_file',
    'surface_damage_label'
]

if 'label_confidence' in all_model_df.columns:
    prediction_columns.append('label_confidence')

patch_predictions = all_model_df[prediction_columns].copy()
patch_predictions['damage_probability'] = all_probability
patch_predictions['predicted_damage'] = (
    patch_predictions['damage_probability'] >= DAMAGE_THRESHOLD
).astype(int)
patch_predictions['prediction_class'] = np.where(
    patch_predictions['predicted_damage'] == 1,
    'damaged',
    'intact',
)

patch_predictions.to_csv(PATCH_PREDICTIONS_PATH, index=False)
display(patch_predictions)
print('Saved:', PATCH_PREDICTIONS_PATH)


## 14. Aggregate to port level

The port-level flag is conservative: if at least one relevant flat-surface patch exceeds the selected damage threshold, the surface is flagged as potential damage. Mean probability, maximum probability and damaged-patch fraction are retained for interpretation.


In [ ]:
port_surface_results = (
    patch_predictions
    .groupby(['source_port_id', 'port_name', 'country', 'disaster'], as_index=False)
    .agg(
        mean_damage_probability=('damage_probability', 'mean'),
        max_damage_probability=('damage_probability', 'max'),
        damaged_patch_fraction=('predicted_damage', 'mean'),
        number_surface_patches=('predicted_damage', 'size'),
    )
)

port_surface_results['surface_damage_detected'] = (
    port_surface_results['max_damage_probability'] >= DAMAGE_THRESHOLD
)

port_surface_results['surface_status'] = np.where(
    port_surface_results['surface_damage_detected'],
    'potential_surface_damage',
    'surface_clear',
)

port_surface_results.to_csv(PORT_RESULTS_PATH, index=False)
display(port_surface_results)
print('Saved:', PORT_RESULTS_PATH)


## 15. Visualise damage probability across each port grid


In [ ]:
for port_id, port_frame in patch_predictions.groupby('source_port_id'):
    probability_grid = (
        port_frame
        .pivot_table(
            index='north_offset_m',
            columns='east_offset_m',
            values='damage_probability',
            aggfunc='mean',
        )
        .sort_index(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(5.5, 5))
    image = ax.imshow(probability_grid.values, vmin=0, vmax=1)

    for row in range(probability_grid.shape[0]):
        for column in range(probability_grid.shape[1]):
            value = probability_grid.iloc[row, column]
            if np.isfinite(value):
                ax.text(column, row, f'{value:.2f}', ha='center', va='center')

    ax.set_xticks(range(probability_grid.shape[1]))
    ax.set_xticklabels([int(value) for value in probability_grid.columns])
    ax.set_yticks(range(probability_grid.shape[0]))
    ax.set_yticklabels([int(value) for value in probability_grid.index])
    ax.set_xlabel('East offset from port centre (m)')
    ax.set_ylabel('North offset from port centre (m)')
    ax.set_title(f'Damage probability: {port_id}')

    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label('Damage probability')
    fig.tight_layout()

    output_path = CNN_DIR / f'{port_id}_damage_probability_grid.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()


## 16. Save the final model and training configuration


In [ ]:
damage_cnn.save(MODEL_PATH)

training_config = {
    'seed': SEED,
    'bands': BAND_NAMES,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'damage_threshold': DAMAGE_THRESHOLD,
    'training_ports': sorted(train_df['source_port_id'].unique().tolist()),
    'validation_ports': sorted(val_df['source_port_id'].unique().tolist()),
    'test_ports': sorted(test_df['source_port_id'].unique().tolist()),
}

with open(CNN_DIR / 'cnn_training_config.json', 'w') as file:
    json.dump(training_config, file, indent=2)

print('Saved model:', MODEL_PATH)


## 17. Inference on new port patches

This function is used after the final model has been trained. New patches must be created by the same six-band Sentinel-1 preprocessing pipeline.


In [ ]:
def predict_new_port_patches(
    metadata_csv,
    patch_directory,
    model_path=MODEL_PATH,
    scaling_path=SCALING_PATH,
    threshold_path=THRESHOLD_PATH,
):
    metadata_csv = Path(metadata_csv)
    patch_directory = Path(patch_directory)

    new_metadata = pd.read_csv(metadata_csv)

    with open(scaling_path) as file:
        scaling = json.load(file)

    with open(threshold_path) as file:
        threshold_config = json.load(file)

    inference_low = np.asarray(scaling['p02'], dtype=np.float32)
    inference_high = np.asarray(scaling['p98'], dtype=np.float32)
    inference_threshold = float(threshold_config['damage_threshold'])
    inference_model = tf.keras.models.load_model(model_path)

    def load_for_inference(raster_path):
        with rasterio.open(raster_path) as src:
            raster = src.read().astype(np.float32)
            nodata = src.nodata

        if raster.shape[0] != N_CHANNELS:
            raise ValueError(
                f'{raster_path} does not contain {N_CHANNELS} bands.'
            )

        raster = np.transpose(raster, (1, 2, 0))
        valid = np.isfinite(raster)
        if nodata is not None:
            valid &= raster != nodata

        scaled = np.zeros_like(raster, dtype=np.float32)

        for band_index in range(N_CHANNELS):
            band = raster[:, :, band_index]
            scaled_band = (
                (band - inference_low[band_index])
                / (inference_high[band_index] - inference_low[band_index])
            )
            scaled_band = np.clip(scaled_band, 0.0, 1.0)
            scaled[:, :, band_index] = np.where(
                valid[:, :, band_index], scaled_band, 0.0
            )

        return scaled

    image_array = np.stack([
        load_for_inference(patch_directory / filename)
        for filename in new_metadata['image_file']
    ])

    probabilities = inference_model.predict(
        image_array,
        batch_size=BATCH_SIZE,
        verbose=0,
    ).ravel()

    output = new_metadata.copy()
    output['damage_probability'] = probabilities
    output['predicted_damage'] = (
        probabilities >= inference_threshold
    ).astype(int)

    return output


print('Inference function ready.')


## 18. Combine surface condition with access results

The CNN is one component of port usability. The final decision should combine surface condition with the bathymetry/access result for the required vessel class.


In [ ]:
def combine_surface_and_access(surface_results, access_results):
    required_columns = {
        'source_port_id',
        'vessel_class',
        'sufficient_depth',
    }

    missing = required_columns - set(access_results.columns)
    if missing:
        raise ValueError(
            f'Bathymetry/access results are missing: {sorted(missing)}'
        )

    combined = access_results.merge(
        surface_results[
            [
                'source_port_id',
                'surface_status',
                'surface_damage_detected',
                'mean_damage_probability',
                'max_damage_probability',
                'damaged_patch_fraction',
            ]
        ],
        on='source_port_id',
        how='left',
    )

    combined['port_usable'] = (
        (combined['surface_damage_detected'] == False)
        & (combined['sufficient_depth'] == True)
    )

    combined['decision'] = np.where(
        combined['port_usable'],
        'usable',
        'not_cleared',
    )

    return combined


print('Surface/access integration function ready.')
